# Bài 2: Hệ tư vấn phim (Movie Recommender System) bằng k-NN
Ứng dụng thuật toán láng giềng gần nhất (k-Nearest Neighbors) để gợi ý các bộ phim tương tự dựa trên đánh giá của người dùng.

---

### 1. Khai báo thư viện và nạp dữ liệu
Sử dụng thư viện `pandas` để xử lý dữ liệu và `sklearn.neighbors` cho mô hình KNN.

In [8]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix
import os

# Đường dẫn dữ liệu từ thư mục /data do thầy cung cấp
try:
    # Giả định file là movies.csv và ratings.csv trong thư mục data
    movies_path = os.path.join('data', 'movies.csv')
    ratings_path = os.path.join('data', 'ratings.csv')
    
    movies = pd.read_csv(movies_path)
    ratings = pd.read_csv(ratings_path)
    print("Dữ liệu nạp thành công từ thư mục data/")
except FileNotFoundError:
    
    # Dữ liệu dự phòng nếu chưa có file local
    movies_url = 'https://raw.githubusercontent.com/sidooms/MovieTweetings/master/latest/movies.dat'
    ratings_url = 'https://raw.githubusercontent.com/sidooms/MovieTweetings/master/latest/ratings.dat'
    
    movies = pd.read_csv(movies_url, sep='::', engine='python', names=['movie_id', 'title', 'genre'], encoding='latin-1')
    ratings = pd.read_csv(ratings_url, sep='::', engine='python', names=['user_id', 'movie_id', 'rating', 'timestamp'])

display(movies.head())
display(ratings.head())

,movie_id,title,genre
0,8,Edison Kinetoscopic Record of a Sneeze (1894),Documentary|Short
1,10,La sortie des usines LumiÃ¨re (1895),Documentary|Short
2,12,The Arrival of a Train (1896),Documentary|Short
3,25,The Oxford and Cambridge University Boat Race ...,NaN
4,91,Le manoir du diable (1896),Short|Horror


,user_id,movie_id,rating,timestamp
0,1,114508,8,1381006850
1,2,499549,9,1376753198
2,2,1305591,8,1376742507
3,2,1428538,1,1371307089
4,3,75314,1,1595468524


### 2. Tiền xử lý dữ liệu
Chuyển đổi dữ liệu sang dạng bảng Pivot (User-Item Matrix) và tối ưu hóa bằng Sparse Matrix.

In [9]:
# Lọc bớt dữ liệu để tránh lỗi tràn bộ nhớ (chỉ lấy phim có >50 đánh giá)
movie_counts = ratings['movie_id'].value_counts()
popular_movies = movie_counts[movie_counts >= 50].index
ratings_filtered = ratings[ratings['movie_id'].isin(popular_movies)]

# Tạo bảng Pivot: Dòng là phim, cột là người dùng, giá trị là điểm đánh giá
movie_user_table = ratings_filtered.pivot(index='movie_id', columns='user_id', values='rating').fillna(0)

# Chuyển sang Ma trận thưa (Sparse Matrix) để tiết kiệm bộ nhớ
movie_user_sparse = csr_matrix(movie_user_table.values)

print(f"Ma trận dữ liệu đã sẵn sàng với hình dạng: {movie_user_table.shape}")

Ma trận dữ liệu đã sẵn sàng với hình dạng: (2983, 65434)


### 3. Huấn luyện mô hình k-NN
Sử dụng khoảng cách Cosine để tìm các bộ phim có sự tương đồng về phong cách đánh giá của người dùng.

In [10]:
# Khởi tạo mô hình KNN với metric 'cosine'
model_knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=20)
model_knn.fit(movie_user_sparse)

print("Mô hình KNN đã được huấn luyện xong!")

Mô hình KNN đã được huấn luyện xong!


### 4. Hàm gợi ý phim (Recommendation Function)
Tìm kiếm tên phim và trả về danh sách các láng giềng gần nhất.

In [11]:
def get_movie_recommendations(movie_name, n_recs=5):
    # Tìm phim khớp với tên nhập vào
    idx = movies[movies['title'].str.contains(movie_name, case=False, na=False)]
    if idx.empty:
        return "Không tìm thấy phim này trong dữ liệu."
    
    m_id = idx.iloc[0]['movie_id']
    
    # Lấy vị trí dòng trong bảng pivot
    if m_id not in movie_user_table.index:
        return "Phim này chưa đủ dữ liệu đánh giá để gợi ý."
        
    movie_idx = movie_user_table.index.get_loc(m_id)
    
    # Tìm kiếm láng giềng
    distances, indices = model_knn.kneighbors(movie_user_sparse[movie_idx], n_neighbors=n_recs+1)
    
    print(f"Vì bạn đã xem '{idx.iloc[0]['title']}', chúng tôi gợi ý:")
    for i in range(1, len(distances.flatten())):
        suggest_id = movie_user_table.index[indices.flatten()[i]]
        suggest_name = movies[movies['movie_id'] == suggest_id]['title'].values[0]
        print(f"{i}. {suggest_name} (Độ tương đồng: {1 - distances.flatten()[i]:.2f})")

# Thử nghiệm gợi ý
get_movie_recommendations('Toy Story', 5)

Vì bạn đã xem 'Toy Story (1995)', chúng tôi gợi ý:
1. Toy Story 2 (1999) (Độ tương đồng: 0.64)
2. Toy Story 3 (2010) (Độ tương đồng: 0.48)
3. Monsters, Inc. (2001) (Độ tương đồng: 0.27)
4. The Lion King (1994) (Độ tương đồng: 0.22)
5. Up (2009) (Độ tương đồng: 0.21)
